# PyMIF beginner notebook 08 - metadata updates, subsetting, pyramids, channels, visualization

This notebook covers day-to-day operations after a dataset is loaded:

- update channel names and colors,
- reorder channels,
- subset T/C/Z/Y/X,
- rebuild the pyramid,
- write a processed copy,
- visualize with napari when installed.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

## Start with a small manager

In [ ]:
manager = make_small_raw_manager(seed=40, num_levels=2)
summarize_manager(manager, "original manager")

## Update metadata safely

Channel metadata must match the number of channels. Colors can be six-digit hex strings or matplotlib color names.

In [ ]:
manager.update_metadata({
    "channel_names": ["Hoechst", "EGFP", "mCherry"],
    "channel_colors": ["blue", "lime", "red"],
    "time_increment": 120.0,
    "time_increment_unit": "second",
})
print(manager.metadata["channel_names"])
print(manager.metadata["channel_colors"])

## Reorder channels

The `new_order` list is a permutation of channel indices. For example `[2, 1, 0]` reverses a 3-channel dataset.

In [ ]:
manager.reorder_channels([2, 1, 0])
print(manager.metadata["channel_names"])
print(manager.data[0].shape)

## Subset the dataset

Subsetting keeps dimensions instead of dropping them. For example `C=[0, 2]` keeps a channel axis of length 2, and `T=0` keeps a time axis of length 1.

Selections must be uniformly spaced when they change scale metadata, for example `Z=[0, 2]` is OK, but `Z=[0, 1, 3]` is not.

In [ ]:
subset = make_small_raw_manager(seed=41, num_levels=2)
subset.subset_dataset(
    T=[0],
    C=[0, 2],
    Z=[0, 2],
    Y=slice(8, 56, 2),
    X=slice(4, 60, 2),
)
summarize_manager(subset, "subset manager")

## Rebuild pyramids

Use `downscale_factor=(1, 2, 2)` when you do not want Z downsampled.

In [ ]:
subset.build_pyramid(num_levels=3, downscale_factor=(1, 2, 2))
summarize_manager(subset, "subset with rebuilt pyramid")

## Do the same operations on a Zarr containing groups and labels

`ZarrManager` applies `subset_dataset`, `build_pyramid`, and `update_metadata` across raw, image groups, and labels by default. Channel operations skip labels because labels usually do not have a channel axis.

In [ ]:
path = clean_path(OUTPUT_DIR / "ops_on_zarr.zarr")
manager = make_small_raw_manager(seed=42, num_levels=2)
manager.to_zarr(path, ngff_version="0.5", zarr_format=3)
z = mm.ZarrManager(path, mode="r+")

z.update_metadata({"channel_names": ["A", "B", "C"], "channel_colors": ["cyan", "magenta", "yellow"]})
z.reorder_channels([1, 2, 0])
z.subset_dataset(T=[0], C=[0, 1], Z=slice(0, 4, 2), Y=slice(0, 64, 2), X=slice(0, 64, 2))
z.build_pyramid(num_levels=2, downscale_factor=(1, 2, 2))
summarize_manager(z, "modified zarr root in memory")

## Write the modified Zarr to a new store

After in-memory transforms such as subsetting, write a new store rather than expecting the old on-disk arrays to change automatically.

In [ ]:
modified_path = clean_path(OUTPUT_DIR / "ops_on_zarr_modified_copy.zarr")
z.to_zarr(modified_path, include_groups=True, include_labels=True, ngff_version="0.5", zarr_format=3)
modified = mm.ZarrManager(modified_path, mode="r")
summarize_manager(modified, "modified zarr read back")

## Optional visualization with napari

`visualize()` opens one manager. `visualize_zarr()` opens root and image groups from a Zarr. These cells require napari and a GUI-capable environment.

In [ ]:
# Optional interactive visualization. Uncomment in a local GUI Python session.
# viewer = manager.visualize(start_level=0, in_memory=False)
# viewer = modified.visualize_zarr()